In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

DB_PATH = PROJECT_ROOT / "database" / "sonoran_cycles.db"

print(DB_PATH)
print(DB_PATH.exists())

/Users/samuellettes/Desktop/Portfolio Project/Supply Chain Analyst/sonoran-cycles-analytics/database/sonoran_cycles.db
True


In [2]:
def run_query(query):
    """
    Runs a SQL query against the Sonoran Cycles SQLite database.
    """

    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(query, conn)

In [3]:
query = """
SELECT
    name AS table_name
FROM sqlite_master
WHERE type = 'table'
ORDER BY
    name;
"""

run_query(query)

,table_name
0,calendar
1,customers
2,daily_kpi_summary
3,daily_order_summary
4,forecast_accuracy_by_model
5,forecast_history
6,inventory_history
7,inventory_kpi_summary
8,model_performance_summary
9,monthly_sales_summary


In [4]:
query = """
SELECT
    model_name,
    category,
    SUM(requested_qty) AS requested_units,
    SUM(fulfilled_qty) AS fulfilled_units,
    SUM(backordered_qty) AS backordered_units,
    ROUND(SUM(extended_price), 2) AS booked_revenue,
    ROUND(SUM(fulfilled_revenue), 2) AS fulfilled_revenue,
    ROUND(
        CAST(SUM(fulfilled_qty) AS FLOAT) / NULLIF(SUM(requested_qty), 0),
        3
    ) AS service_level,
    ROUND(
        CAST(SUM(backordered_qty) AS FLOAT) / NULLIF(SUM(requested_qty), 0),
        3
    ) AS backorder_rate
FROM sales_order_lines
GROUP BY
    model_name,
    category
ORDER BY
    booked_revenue DESC;
"""

model_performance = run_query(query)
model_performance

,model_name,category,requested_units,fulfilled_units,backordered_units,booked_revenue,fulfilled_revenue,service_level,backorder_rate
0,Romero,Aggressive Trail,51613,42035,9578,1.715949e+08,1.396967e+08,0.814,0.186
1,Sabino,Trail,54391,43932,10459,1.640263e+08,1.323868e+08,0.808,0.192
2,Oracle,Enduro,35759,32535,3224,1.411246e+08,1.283495e+08,0.910,0.090
3,Catalina,Cross Country,28566,24962,3604,6.280795e+07,5.486534e+07,0.874,0.126
4,Sky Island,eMTB,11986,11938,48,5.876006e+07,5.852482e+07,0.996,0.004
5,Rincon,Downcountry,16454,15685,769,4.181898e+07,3.987979e+07,0.953,0.047
6,Sonoita,Gravel,7605,7587,18,1.855156e+07,1.850667e+07,0.998,0.002
